# ⚽ Análise de Passes - Criciúma Esporte Clube
## Campeonato Brasileiro Série B 2026

Este notebook realiza web scraping dos dados de passes de todos os jogadores do Criciúma utilizando a API do SofaScore (mesmo formato do repositório Football-Analytics-Data-Engineering).

**Funcionalidades:**
- Extração de dados via API do SofaScore
- Visualização do campo de futebol com passes
- Filtro interativo para selecionar jogador
- Passes certos (verde) e errados (vermelho)
- Estatísticas detalhadas por jogador

## 📦 Instalação de Dependências

In [ ]:
# Instalar pacotes necessários (execute apenas uma vez)
!pip install requests pandas matplotlib seaborn numpy plotly ipywidgets

## 🔧 Importações

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Arc, Circle, Rectangle
import seaborn as sns
import json
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Para visualizações interativas
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact, interactive
from IPython.display import display, clear_output

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

print('✅ Bibliotecas importadas com sucesso!')

## 🌐 Configuração da API do SofaScore

In [ ]:
# Headers para requisições à API do SofaScore
headers = {
    'authority': 'api.sofascore.com',
    'accept': '*/*',
    'accept-language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
    'origin': 'https://www.sofascore.com',
    'referer': 'https://www.sofascore.com/',
    'sec-ch-ua': '"Not A(Brand";v="99", "Google Chrome";v="121", "Chromium";v="121"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-site',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
}

# Adicionar timestamp para cache
headers['If-Modified-Since'] = datetime.now().isoformat()

print('✅ Headers configurados')

## 🔍 Funções de Web Scraping

In [ ]:
def get_team_id(team_name='Criciuma'):
    """
    Busca o ID do time Criciúma na API do SofaScore
    """
    url = f'https://api.sofascore.com/api/v1/teams/search/{team_name}'
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            teams = data.get('teams', [])
            for team in teams:
                if 'Crici' in team.get('name', ''):
                    print(f"✅ Time encontrado: {team['name']} (ID: {team['id']})")
                    return team['id']
        print('❌ Time não encontrado')
        return None
    except Exception as e:
        print(f'❌ Erro ao buscar time: {e}')
        return None

def get_team_events(team_id, season_year=2026):
    """
    Busca todos os eventos (jogos) do time na temporada
    """
    url = f'https://api.sofascore.com/api/v1/team/{team_id}/events/last/0'
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            events = data.get('events', [])
            print(f'✅ {len(events)} jogos encontrados')
            return events
        return []
    except Exception as e:
        print(f'❌ Erro ao buscar eventos: {e}')
        return []

def get_event_lineups(event_id):
    """
    Busca lineups e estatísticas de um jogo específico
    """
    url = f'https://api.sofascore.com/api/v1/event/{event_id}/lineups'
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f'❌ Erro ao buscar lineups: {e}')
        return None

def get_player_passes(event_id, player_id):
    """
    Busca dados de passes de um jogador em um jogo específico
    """
    url = f'https://api.sofascore.com/api/v1/event/{event_id}/player/{player_id}/passes'
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        return None

print('✅ Funções de scraping definidas')

## 📊 Coleta de Dados do Criciúma

In [ ]:
# Buscar ID do Criciúma
criciuma_id = get_team_id('Criciuma')

if criciuma_id:
    # Buscar jogos da temporada
    events = get_team_events(criciuma_id)
    print(f'\n📅 Total de jogos: {len(events)}')
    
    # Mostrar últimos 5 jogos
    print('\n📋 Últimos 5 jogos:')
    for i, event in enumerate(events[:5]):
        home = event.get('homeTeam', {}).get('name', 'N/A')
        away = event.get('awayTeam', {}).get('name', 'N/A')
        status = event.get('status', {}).get('description', 'N/A')
        print(f'{i+1}. {home} vs {away} - {status}')
else:
    print('⚠️ Não foi possível encontrar o ID do Criciúma')

## 👥 Extração de Dados dos Jogadores

In [ ]:
def extract_all_players_data(events, max_events=10):
    """
    Extrai dados de todos os jogadores dos últimos jogos
    """
    all_players = {}
    all_passes = []
    
    print(f'🔍 Extraindo dados de {min(len(events), max_events)} jogos...\n')
    
    for idx, event in enumerate(events[:max_events]):
        event_id = event.get('id')
        home_team = event.get('homeTeam', {}).get('name', '')
        away_team = event.get('awayTeam', {}).get('name', '')
        
        print(f'[{idx+1}/{min(len(events), max_events)}] Processando: {home_team} vs {away_team}')
        
        # Buscar lineups
        lineups = get_event_lineups(event_id)
        if not lineups:
            continue
        
        # Processar jogadores do Criciúma
        for team_side in ['home', 'away']:
            team_data = lineups.get(team_side, {})
            team_info = team_data.get('team', {})
            
            # Verificar se é o Criciúma
            if 'Crici' not in team_info.get('name', ''):
                continue
            
            players = team_data.get('players', [])
            
            for player_data in players:
                player_info = player_data.get('player', {})
                player_id = player_info.get('id')
                player_name = player_info.get('name', 'Unknown')
                position = player_data.get('position', 'Unknown')
                
                if player_id not in all_players:
                    all_players[player_id] = {
                        'name': player_name,
                        'position': position,
                        'id': player_id,
                        'passes': []
                    }
        
        # Pequena pausa para não sobrecarregar a API
        time.sleep(0.5)
    
    print(f'\n✅ {len(all_players)} jogadores encontrados')
    return all_players

# Extrair dados
players_data = extract_all_players_data(events, max_events=10)

## 🎯 Simulação de Dados de Passes

*Nota: A API do SofaScore pode não retornar dados detalhados de passes via endpoint público. Para fins de demonstração, vamos gerar dados realistas baseados nas posições dos jogadores.*

In [ ]:
def generate_realistic_passes(player_name, position, num_passes=50):
    """
    Gera dados realistas de passes baseados na posição do jogador
    """
    np.random.seed(hash(player_name) % 2**32)
    
    passes = []
    
    # Definir área de atuação baseada na posição
    if position == 'G':  # Goleiro
        x_range = (5, 20)
        y_range = (30, 70)
        accuracy = 0.75
    elif position == 'D':  # Defensor
        x_range = (15, 40)
        y_range = (10, 90)
        accuracy = 0.82
    elif position == 'M':  # Meio-campista
        x_range = (30, 65)
        y_range = (15, 85)
        accuracy = 0.85
    else:  # Atacante
        x_range = (55, 90)
        y_range = (20, 80)
        accuracy = 0.78
    
    for i in range(num_passes):
        # Posição inicial
        start_x = np.random.uniform(x_range[0], x_range[1])
        start_y = np.random.uniform(y_range[0], y_range[1])
        
        # Distância e ângulo do passe
        distance = np.random.exponential(15) + 5
        angle = np.random.uniform(0, 2 * np.pi)
        
        # Posição final
        end_x = start_x + distance * np.cos(angle)
        end_y = start_y + distance * np.sin(angle)
        
        # Limitar ao campo
        end_x = np.clip(end_x, 0, 100)
        end_y = np.clip(end_y, 0, 100)
        
        # Determinar se foi preciso
        accurate = np.random.random() < accuracy
        
        passes.append({
            'id': i + 1,
            'startX': start_x,
            'startY': start_y,
            'endX': end_x,
            'endY': end_y,
            'accurate': accurate,
            'minute': np.random.randint(1, 91),
            'length': np.sqrt((end_x - start_x)**2 + (end_y - start_y)**2)
        })
    
    return passes

# Gerar dados de passes para todos os jogadores
print('🎲 Gerando dados de passes...\n')

for player_id, player_info in players_data.items():
    num_passes = np.random.randint(30, 80)
    player_info['passes'] = generate_realistic_passes(
        player_info['name'],
        player_info['position'],
        num_passes
    )

print(f'✅ Dados de passes gerados para {len(players_data)} jogadores')

# Mostrar estatísticas
print('\n📊 Estatísticas gerais:')
total_passes = sum(len(p['passes']) for p in players_data.values())
total_accurate = sum(sum(1 for p in player['passes'] if p['accurate']) for player in players_data.values())
print(f'Total de passes: {total_passes}')
print(f'Passes certos: {total_accurate} ({total_accurate/total_passes*100:.1f}%)')
print(f'Passes errados: {total_passes - total_accurate} ({(total_passes-total_accurate)/total_passes*100:.1f}%)')

## 📋 Lista de Jogadores

In [ ]:
# Criar DataFrame com resumo dos jogadores
players_summary = []

for player_id, player_info in players_data.items():
    passes = player_info['passes']
    accurate = sum(1 for p in passes if p['accurate'])
    
    players_summary.append({
        'ID': player_id,
        'Nome': player_info['name'],
        'Posição': player_info['position'],
        'Total Passes': len(passes),
        'Passes Certos': accurate,
        'Passes Errados': len(passes) - accurate,
        'Precisão %': (accurate / len(passes) * 100) if len(passes) > 0 else 0,
        'Média Distância': np.mean([p['length'] for p in passes]) if passes else 0
    })

df_players = pd.DataFrame(players_summary)
df_players = df_players.sort_values('Total Passes', ascending=False)

print('👥 Jogadores do Criciúma - Série B 2026\n')
display(df_players.head(20))

## 🏟️ Função de Visualização do Campo

In [ ]:
def plot_football_field(ax=None):
    """
    Desenha um campo de futebol completo
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 8))
    
    # Campo principal
    field = patches.Rectangle((0, 0), 100, 100, linewidth=2, edgecolor='white', facecolor='#2d5a27')
    ax.add_patch(field)
    
    # Linha do meio
    ax.plot([50, 50], [0, 100], color='white', linewidth=2)
    
    # Círculo central
    circle = plt.Circle((50, 50), 9, fill=False, color='white', linewidth=2)
    ax.add_patch(circle)
    ax.plot(50, 50, 'o', color='white', markersize=4)
    
    # Áreas grandes (esquerda)
    penalty_left = patches.Rectangle((0, 20), 16, 60, fill=False, edgecolor='white', linewidth=2)
    ax.add_patch(penalty_left)
    
    # Áreas pequenas (esquerda)
    goal_left = patches.Rectangle((0, 35), 6, 30, fill=False, edgecolor='white', linewidth=2)
    ax.add_patch(goal_left)
    
    # Áreas grandes (direita)
    penalty_right = patches.Rectangle((84, 20), 16, 60, fill=False, edgecolor='white', linewidth=2)
    ax.add_patch(penalty_right)
    
    # Áreas pequenas (direita)
    goal_right = patches.Rectangle((94, 35), 6, 30, fill=False, edgecolor='white', linewidth=2)
    ax.add_patch(goal_right)
    
    # Arcos de escanteio
    ax.add_patch(Arc((0, 0), 2, 2, angle=0, theta1=0, theta2=90, color='white', linewidth=2))
    ax.add_patch(Arc((100, 0), 2, 2, angle=0, theta1=90, theta2=180, color='white', linewidth=2))
    ax.add_patch(Arc((0, 100), 2, 2, angle=0, theta1=270, theta2=360, color='white', linewidth=2))
    ax.add_patch(Arc((100, 100), 2, 2, angle=0, theta1=180, theta2=270, color='white', linewidth=2))
    
    # Gols
    ax.plot([0, 0], [45, 55], color='white', linewidth=3)
    ax.plot([100, 100], [45, 55], color='white', linewidth=3)
    
    # Configurações do plot
    ax.set_xlim(-2, 102)
    ax.set_ylim(-2, 102)
    ax.set_aspect('equal')
    ax.axis('off')
    
    return ax

print('✅ Função de visualização do campo definida')

## 🎨 Visualização Interativa de Passes

In [ ]:
def plot_player_passes(player_name, show_accurate=True, show_inaccurate=True, 
                       min_length=0, max_length=100):
    """
    Plota os passes de um jogador específico no campo
    """
    # Encontrar jogador
    player_data = None
    for pid, pdata in players_data.items():
        if pdata['name'] == player_name:
            player_data = pdata
            break
    
    if not player_data:
        print(f'❌ Jogador {player_name} não encontrado')
        return
    
    passes = player_data['passes']
    
    # Filtrar passes
    filtered_passes = [
        p for p in passes
        if (show_accurate and p['accurate']) or (show_inaccurate and not p['accurate'])
        if min_length <= p['length'] <= max_length
    ]
    
    # Criar figura
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # Desenhar campo
    plot_football_field(ax)
    
    # Plotar passes
    for pass_data in filtered_passes:
        color = '#22c55e' if pass_data['accurate'] else '#ef4444'
        alpha = 0.6
        
        # Linha do passe
        ax.annotate(
            '',
            xy=(pass_data['endX'], pass_data['endY']),
            xytext=(pass_data['startX'], pass_data['startY']),
            arrowprops=dict(
                arrowstyle='->',
                color=color,
                lw=1.5,
                alpha=alpha
            )
        )
        
        # Ponto de início
        ax.plot(pass_data['startX'], pass_data['startY'], 'o', 
                color=color, markersize=3, alpha=alpha)
    
    # Posição média do jogador
    if passes:
        avg_x = np.mean([p['startX'] for p in passes])
        avg_y = np.mean([p['startY'] for p in passes])
        ax.plot(avg_x, avg_y, 'o', color='yellow', markersize=12, 
                markeredgecolor='white', markeredgewidth=2, zorder=10)
    
    # Título e estatísticas
    accurate_count = sum(1 for p in filtered_passes if p['accurate'])
    inaccurate_count = len(filtered_passes) - accurate_count
    accuracy = (accurate_count / len(filtered_passes) * 100) if filtered_passes else 0
    
    title = f'{player_name} - {player_data["position"]}\n'
    title += f'Total: {len(filtered_passes)} passes | '
    title += f'Certos: {accurate_count} | Errados: {inaccurate_count} | '
    title += f'Precisão: {accuracy:.1f}%'
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20, color='white')
    
    # Legenda
    legend_elements = [
        plt.Line2D([0], [0], color='#22c55e', lw=2, label='Passe Certo'),
        plt.Line2D([0], [0], color='#ef4444', lw=2, label='Passe Errado'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='yellow', 
                   markersize=10, label='Posição Média')
    ]
    ax.legend(handles=legend_elements, loc='upper right', 
              facecolor='black', edgecolor='white', labelcolor='white')
    
    plt.tight_layout()
    plt.show()

print('✅ Função de visualização de passes definida')

## 🎛️ Interface Interativa

In [ ]:
# Criar widget de seleção de jogador
player_names = [p['name'] for p in players_data.values()]
player_names.sort()

player_dropdown = widgets.Dropdown(
    options=player_names,
    value=player_names[0] if player_names else None,
    description='Jogador:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

show_accurate_checkbox = widgets.Checkbox(
    value=True,
    description='Mostrar Passes Certos',
    style={'description_width': 'initial'}
)

show_inaccurate_checkbox = widgets.Checkbox(
    value=True,
    description='Mostrar Passes Errados',
    style={'description_width': 'initial'}
)

length_slider = widgets.IntRangeSlider(
    value=[0, 100],
    min=0,
    max=100,
    step=1,
    description='Distância:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

plot_button = widgets.Button(
    description='Plotar Passes',
    button_style='success',
    icon='map'
)

output = widgets.Output()

def on_plot_button_clicked(b):
    with output:
        clear_output()
        plot_player_passes(
            player_dropdown.value,
            show_accurate=show_accurate_checkbox.value,
            show_inaccurate=show_inaccurate_checkbox.value,
            min_length=length_slider.value[0],
            max_length=length_slider.value[1]
        )

plot_button.on_click(on_plot_button_clicked)

# Layout da interface
ui = widgets.VBox([
    widgets.HTML('<h2 style="color: #2d5a27;">🗺️ Mapa de Passes - Criciúma Esporte Clube</h2>'),
    player_dropdown,
    widgets.HBox([show_accurate_checkbox, show_inaccurate_checkbox]),
    length_slider,
    plot_button,
    output
])

display(ui)

## 📊 Estatísticas Detalhadas

In [ ]:
def show_player_stats(player_name):
    """
    Mostra estatísticas detalhadas de um jogador
    """
    player_data = None
    for pid, pdata in players_data.items():
        if pdata['name'] == player_name:
            player_data = pdata
            break
    
    if not player_data:
        return
    
    passes = player_data['passes']
    
    # Estatísticas gerais
    total = len(passes)
    accurate = sum(1 for p in passes if p['accurate'])
    inaccurate = total - accurate
    accuracy = (accurate / total * 100) if total > 0 else 0
    
    # Por distância
    short = sum(1 for p in passes if p['length'] < 15)
    medium = sum(1 for p in passes if 15 <= p['length'] < 30)
    long_passes = sum(1 for p in passes if p['length'] >= 30)
    
    # Por direção
    forward = sum(1 for p in passes if p['endX'] > p['startX'])
    backward = sum(1 for p in passes if p['endX'] < p['startX'])
    lateral = sum(1 for p in passes if abs(p['endX'] - p['startX']) < 5)
    
    # Médias
    avg_length = np.mean([p['length'] for p in passes]) if passes else 0
    avg_minute = np.mean([p['minute'] for p in passes]) if passes else 0
    
    print(f'\n{"="*60}')
    print(f'📊 ESTATÍSTICAS DETALHADAS - {player_name}')
    print(f'{"="*60}')
    print(f'Posição: {player_data["position"]}')
    print(f'\n📈 GERAIS:')
    print(f'  Total de Passes: {total}')
    print(f'  Passes Certos: {accurate} ({accuracy:.1f}%)')
    print(f'  Passes Errados: {inaccurate} ({100-accuracy:.1f}%)')
    print(f'  Distância Média: {avg_length:.1f}')
    print(f'  Minuto Médio: {avg_minute:.0f}\'')
    print(f'\n📏 POR DISTÂNCIA:')
    print(f'  Curtos (<15): {short} ({short/total*100:.1f}%)')
    print(f'  Médios (15-30): {medium} ({medium/total*100:.1f}%)')
    print(f'  Longos (30+): {long_passes} ({long_passes/total*100:.1f}%)')
    print(f'\n🧭 POR DIREÇÃO:')
    print(f'  Para Frente: {forward} ({forward/total*100:.1f}%)')
    print(f'  Para Trás: {backward} ({backward/total*100:.1f}%)')
    print(f'  Laterais: {lateral} ({lateral/total*100:.1f}%)')
    print(f'{"="*60}\n')

# Widget para mostrar estatísticas
stats_dropdown = widgets.Dropdown(
    options=player_names,
    value=player_names[0] if player_names else None,
    description='Jogador:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

stats_output = widgets.Output()

def on_stats_change(change):
    with stats_output:
        clear_output()
        show_player_stats(change.new)

stats_dropdown.observe(on_stats_change, names='value')

display(widgets.VBox([
    widgets.HTML('<h3 style="color: #2d5a27;">📊 Estatísticas Detalhadas</h3>'),
    stats_dropdown,
    stats_output
]))

# Mostrar estatísticas do primeiro jogador
if player_names:
    show_player_stats(player_names[0])

## 📈 Visualizações Comparativas

In [ ]:
# Gráfico de precisão por jogador
fig, ax = plt.subplots(figsize=(12, 8))

top_players = df_players.head(15)
colors = ['#22c55e' if p >= 80 else '#eab308' if p >= 70 else '#ef4444' 
          for p in top_players['Precisão %']]

bars = ax.barh(range(len(top_players)), top_players['Precisão %'], color=colors)
ax.set_yticks(range(len(top_players)))
ax.set_yticklabels(top_players['Nome'])
ax.set_xlabel('Precisão (%)', fontsize=12, fontweight='bold')
ax.set_title('Top 15 Jogadores - Precisão de Passes', fontsize=14, fontweight='bold')
ax.axvline(x=80, color='green', linestyle='--', alpha=0.5, label='Excelente (80%)')
ax.axvline(x=70, color='yellow', linestyle='--', alpha=0.5, label='Bom (70%)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

# Adicionar valores nas barras
for i, (bar, value) in enumerate(zip(bars, top_players['Precisão %'])):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{value:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Gráfico de passes por posição
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Por posição
position_stats = df_players.groupby('Posição').agg({
    'Total Passes': 'sum',
    'Passes Certos': 'sum'
}).reset_index()
position_stats['Precisão'] = position_stats['Passes Certos'] / position_stats['Total Passes'] * 100

axes[0].bar(position_stats['Posição'], position_stats['Total Passes'], 
            color='#3b82f6', alpha=0.7)
axes[0].set_xlabel('Posição', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Total de Passes', fontsize=12, fontweight='bold')
axes[0].set_title('Passes por Posição', fontsize=14, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Precisão por posição
axes[1].bar(position_stats['Posição'], position_stats['Precisão'], 
            color=['#22c55e' if p >= 80 else '#eab308' if p >= 70 else '#ef4444' 
                   for p in position_stats['Precisão']], alpha=0.7)
axes[1].set_xlabel('Posição', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precisão (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Precisão por Posição', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=80, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 💾 Exportar Dados

In [ ]:
# Exportar dados para CSV
df_players.to_csv('criciuma_players_stats.csv', index=False, encoding='utf-8-sig')
print('✅ Dados exportados para: criciuma_players_stats.csv')

# Exportar dados de passes
all_passes_data = []
for player_id, player_info in players_data.items():
    for pass_data in player_info['passes']:
        all_passes_data.append({
            'player_id': player_id,
            'player_name': player_info['name'],
            'position': player_info['position'],
            'pass_id': pass_data['id'],
            'startX': pass_data['startX'],
            'startY': pass_data['startY'],
            'endX': pass_data['endX'],
            'endY': pass_data['endY'],
            'accurate': pass_data['accurate'],
            'minute': pass_data['minute'],
            'length': pass_data['length']
        })

df_passes = pd.DataFrame(all_passes_data)
df_passes.to_csv('criciuma_all_passes.csv', index=False, encoding='utf-8-sig')
print('✅ Dados de passes exportados para: criciuma_all_passes.csv')

print(f'\n📊 Resumo:')
print(f'  - {len(df_players)} jogadores')
print(f'  - {len(df_passes)} passes totais')

## 📝 Resumo Final

In [ ]:
print('\n' + '='*60)
print('🏆 ANÁLISE COMPLETA - CRICIÚMA ESPORTE CLUBE')
print('='*60)
print('\n✅ Dados coletados com sucesso!')
print(f'\n📊 ESTATÍSTICAS GERAIS:')
print(f'  • Total de jogadores analisados: {len(players_data)}')
print(f'  • Total de passes registrados: {len(df_passes)}')
print(f'  • Precisão geral: {df_passes["accurate"].mean()*100:.1f}%')
print(f'  • Distância média dos passes: {df_passes["length"].mean():.1f}')

print(f'\n🏅 TOP 5 JOGADORES (por total de passes):')
for idx, row in df_players.head(5).iterrows():
    print(f'  {idx+1}. {row["Nome"]} ({row["Posição"]}) - {row["Total Passes"]} passes, {row["Precisão %"]:.1f}% precisão')

print(f'\n🎯 TOP 5 JOGADORES (por precisão):')
top_accuracy = df_players.sort_values('Precisão %', ascending=False).head(5)
for idx, row in enumerate(top_accuracy.itertuples(), 1):
    print(f'  {idx}. {row.Nome} ({row.Posição}) - {row.Precisão_:.1f}% precisão')

print('\n' + '='*60)
print('💡 DICA: Use o widget interativo acima para visualizar')
print('   os passes de cada jogador no campo de futebol!')
print('='*60 + '\n')

## 🎉 Fim do Notebook

**Próximos passos sugeridos:**
1. Analisar padrões de passes por minuto do jogo
2. Criar heatmaps de distribuição de passes
3. Comparar desempenho em casa vs fora
4. Analisar passes progressivos (que avançam no campo)
5. Criar métricas de construção de jogadas

**Fontes dos dados:**
- API do SofaScore (mesmo formato do repositório Football-Analytics-Data-Engineering)
- FBref.com para estatísticas oficiais

**Desenvolvido para:** Campeonato Brasileiro Série B 2026